# Midstream Predictive Maintenance — ML Pipeline

**End-to-end machine learning pipeline** for predicting equipment failures and remaining useful life (RUL) across a fleet of 50 pumps and compressors in the Permian Basin.

## Pipeline Overview
1. **Data Loading & EDA** — Load feature store, analyze sensor distributions and failure patterns
2. **Feature Engineering** — Interaction features, deviation metrics, maintenance signals
3. **Class Imbalance Handling** — SMOTE oversampling for minority failure classes
4. **Failure Classifier** — Multi-class XGBoost with hyperparameter tuning via RandomizedSearchCV
5. **RUL Regressor** — XGBoost regressor for remaining useful life estimation
6. **Model Explainability** — SHAP values for feature importance and prediction explanations
7. **Model Registry** — Register both models in Snowflake ML Registry

**Run `score_fleet.ipynb` after this notebook** to load models from the registry and score the fleet.

**Models**: XGBoost (Classifier + Regressor)  
**Target**: 5 failure modes (BEARING_WEAR, VALVE_FAILURE, SEAL_LEAK, SURGE, OVERHEATING) + NORMAL  
**Features**: 29 sensor aggregations + maintenance history + operating hours + engineered features

---
## 1. Environment Setup

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE DATABASE PDM_DEMO').collect()
session.sql('USE SCHEMA ANALYTICS').collect()
session.sql('USE WAREHOUSE PDM_DEMO_WH').collect()
print(f'Snowpark version: {session.version}')
print('Session ready — PDM_DEMO.ANALYTICS')

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, confusion_matrix,
    mean_absolute_error, r2_score, root_mean_squared_error,
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier, XGBRegressor
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.titlesize': 12})

print('Libraries loaded')

---
## 2. Data Loading & Exploratory Analysis

In [ ]:
pdf = session.table('ANALYTICS.FEATURE_STORE').to_pandas()
print(f'Feature Store: {pdf.shape[0]:,} rows × {pdf.shape[1]} columns')
print(f'Assets: {pdf["ASSET_ID"].nunique()}')
print(f'Date range: {pdf["AS_OF_TS"].min()} → {pdf["AS_OF_TS"].max()}')
print(f'\nAsset types:\n{pdf["ASSET_TYPE"].value_counts().to_string()}')
print(f'\nFailure label distribution:\n{pdf["FAILURE_LABEL"].value_counts().to_string()}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Sensor Distributions by Failure Mode', fontweight='bold')

sensors = ['VIBRATION_MEAN_24H', 'TEMPERATURE_MEAN_24H', 'PRESSURE_MEAN_24H',
           'FLOW_RATE_MEAN_24H', 'POWER_DRAW_MEAN_24H', 'RPM_MEAN_24H']
colors = {'NORMAL': '#22c55e', 'BEARING_WEAR': '#ef4444', 'VALVE_FAILURE': '#f59e0b',
          'SEAL_LEAK': '#3b82f6', 'SURGE': '#a855f7', 'OVERHEATING': '#f97316'}

for ax, sensor in zip(axes.flat, sensors):
    for label in pdf['FAILURE_LABEL'].unique():
        subset = pdf[pdf['FAILURE_LABEL'] == label][sensor].dropna()
        if len(subset) > 0:
            ax.hist(subset, bins=30, alpha=0.5, label=label, color=colors.get(label, '#888'))
    ax.set_title(sensor.replace('_MEAN_24H', '').replace('_', ' '))
    ax.set_ylabel('Count')

axes[0, 0].legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
degrading = pdf[pdf['FAILURE_LABEL'] != 'NORMAL']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label in degrading['FAILURE_LABEL'].unique():
    sub = degrading[degrading['FAILURE_LABEL'] == label].sort_values('DAYS_TO_FAILURE', ascending=False)
    axes[0].plot(sub['DAYS_TO_FAILURE'], sub['VIBRATION_MEAN_24H'], 'o-',
                 label=label, color=colors.get(label, '#888'), markersize=4, alpha=0.7)
axes[0].set_xlabel('Days to Failure')
axes[0].set_ylabel('Vibration (mm/s RMS)')
axes[0].set_title('Vibration Trend as Failure Approaches')
axes[0].legend(fontsize=8)
axes[0].invert_xaxis()

for label in degrading['FAILURE_LABEL'].unique():
    sub = degrading[degrading['FAILURE_LABEL'] == label].sort_values('DAYS_TO_FAILURE', ascending=False)
    axes[1].plot(sub['DAYS_TO_FAILURE'], sub['TEMPERATURE_MEAN_24H'], 'o-',
                 label=label, color=colors.get(label, '#888'), markersize=4, alpha=0.7)
axes[1].set_xlabel('Days to Failure')
axes[1].set_ylabel('Temperature (°F)')
axes[1].set_title('Temperature Trend as Failure Approaches')
axes[1].legend(fontsize=8)
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = [c for c in pdf.columns if pdf[c].dtype in ['float64', 'int64'] and c not in ['ASSET_ID', 'LABEL_ENCODED']]
corr = pdf[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels([c.replace('_MEAN_24H', '').replace('_', ' ')[:15] for c in corr.columns], rotation=45, ha='right', fontsize=7)
ax.set_yticklabels([c.replace('_MEAN_24H', '').replace('_', ' ')[:15] for c in corr.columns], fontsize=7)
plt.colorbar(im, label='Correlation')
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

Create interaction features that capture domain-specific degradation signatures:
- **Vibration × Temperature** — bearing wear produces both heat and vibration simultaneously
- **Power efficiency** — power draw relative to flow rate increases as components degrade  
- **Pressure variability** — valve failures cause erratic pressure swings
- **Sensor deviations** — gap between max and mean indicates intermittent spikes

In [ ]:
pdf['IS_PUMP'] = (pdf['ASSET_TYPE'] == 'PUMP').astype(int)

pdf['VIB_TEMP_INTERACTION'] = pdf['VIBRATION_MEAN_24H'] * pdf['TEMPERATURE_MEAN_24H'] / 1000
pdf['POWER_EFFICIENCY'] = pdf['POWER_DRAW_MEAN_24H'] / (pdf['FLOW_RATE_MEAN_24H'] + 1)
pdf['PRESSURE_VARIABILITY'] = pdf['PRESSURE_STD_24H'] / (pdf['PRESSURE_MEAN_24H'] + 1)
pdf['VIB_DEVIATION'] = pdf['VIBRATION_MAX_24H'] - pdf['VIBRATION_MEAN_24H']
pdf['TEMP_DEVIATION'] = pdf['TEMPERATURE_MAX_24H'] - pdf['TEMPERATURE_MEAN_24H']
pdf['MAINT_RECENCY_SCORE'] = 1 / (pdf['DAYS_SINCE_MAINTENANCE'] + 1)

normal_mask = pdf['FAILURE_LABEL'] == 'NORMAL'
VIB_NORMAL_MEAN = pdf.loc[normal_mask, 'VIBRATION_MEAN_24H'].mean()
VIB_NORMAL_STD = max(pdf.loc[normal_mask, 'VIBRATION_MEAN_24H'].std(), 0.01)
TEMP_NORMAL_MEAN = pdf.loc[normal_mask, 'TEMPERATURE_MEAN_24H'].mean()
TEMP_NORMAL_STD = max(pdf.loc[normal_mask, 'TEMPERATURE_MEAN_24H'].std(), 0.01)

pdf['VIB_SEVERITY'] = (pdf['VIBRATION_MEAN_24H'] - VIB_NORMAL_MEAN) / VIB_NORMAL_STD
pdf['TEMP_SEVERITY'] = (pdf['TEMPERATURE_MEAN_24H'] - TEMP_NORMAL_MEAN) / TEMP_NORMAL_STD
pdf['DEGRADATION_INTENSITY'] = np.sqrt(pdf['VIB_SEVERITY'].clip(0)**2 + pdf['TEMP_SEVERITY'].clip(0)**2)

print(f'Normal baselines: VIB={VIB_NORMAL_MEAN:.2f}+/-{VIB_NORMAL_STD:.2f}, TEMP={TEMP_NORMAL_MEAN:.2f}+/-{TEMP_NORMAL_STD:.2f}')

FEATURE_COLS = [
    'VIBRATION_MEAN_24H', 'VIBRATION_STD_24H', 'VIBRATION_MAX_24H', 'VIBRATION_TREND',
    'TEMPERATURE_MEAN_24H', 'TEMPERATURE_STD_24H', 'TEMPERATURE_MAX_24H', 'TEMPERATURE_TREND',
    'PRESSURE_MEAN_24H', 'PRESSURE_STD_24H', 'FLOW_RATE_MEAN_24H',
    'RPM_MEAN_24H', 'RPM_STD_24H', 'POWER_DRAW_MEAN_24H',
    'DAYS_SINCE_MAINTENANCE', 'MAINTENANCE_COUNT_90D', 'OPERATING_HOURS',
    'DIFF_PRESSURE_MEAN_24H', 'SEAL_TEMP_MEAN_24H',
    'DISCHARGE_TEMP_MEAN_24H', 'COMPRESSION_RATIO_MEAN', 'OIL_PRESSURE_MEAN_24H',
    'IS_PUMP',
    'VIB_TEMP_INTERACTION', 'POWER_EFFICIENCY', 'PRESSURE_VARIABILITY',
    'VIB_DEVIATION', 'TEMP_DEVIATION', 'MAINT_RECENCY_SCORE',
    'VIB_SEVERITY', 'TEMP_SEVERITY', 'DEGRADATION_INTENSITY',
]

for col in FEATURE_COLS:
    if col in pdf.columns:
        pdf[col] = pdf[col].fillna(0)

print(f'Total features: {len(FEATURE_COLS)}')
print(f'Severity features: VIB_SEVERITY, TEMP_SEVERITY, DEGRADATION_INTENSITY')

---
## 4. Train/Test Split & Class Imbalance Handling

The dataset is heavily imbalanced (~98% NORMAL). We use **SMOTE** (Synthetic Minority Over-sampling Technique) to generate synthetic examples of failure modes, giving the classifier enough signal to learn degradation patterns without losing the majority class distribution during evaluation.

In [ ]:
le = LabelEncoder()
pdf['LABEL_ENCODED'] = le.fit_transform(pdf['FAILURE_LABEL'])
print(f'Classes: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X = pdf[FEATURE_COLS].values
y_cls = pdf['LABEL_ENCODED'].values
y_rul = pdf['DAYS_TO_FAILURE'].values

X_train, X_test, y_train_cls, y_test_cls, y_train_rul, y_test_rul = train_test_split(
    X, y_cls, y_rul, test_size=0.2, random_state=42, stratify=y_cls
)

print(f'\nTrain: {X_train.shape[0]:,} samples')
print(f'Test:  {X_test.shape[0]:,} samples')
print(f'\nTrain class distribution:')
for cls_idx, cls_name in enumerate(le.classes_):
    count = (y_train_cls == cls_idx).sum()
    print(f'  {cls_name}: {count} ({count/len(y_train_cls)*100:.1f}%)')

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=2)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train_cls)

print(f'Before SMOTE: {X_train.shape[0]:,} samples')
print(f'After SMOTE:  {X_train_bal.shape[0]:,} samples')
print(f'\nBalanced class distribution:')
for cls_idx, cls_name in enumerate(le.classes_):
    count = (y_train_bal == cls_idx).sum()
    print(f'  {cls_name}: {count}')

---
## 5. Failure Classifier — XGBoost with Hyperparameter Tuning

We use `RandomizedSearchCV` with stratified 5-fold cross-validation to find the optimal hyperparameters for the multi-class failure classifier. This searches across 9 hyperparameter dimensions (learning rate, tree depth, regularization, sampling, etc.).

In [ ]:
clf = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=3.0,
    objective='multi:softprob',
    num_class=len(le.classes_),
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist',
)

print('Training classifier (XGBoost)...')
clf.fit(X_train_bal, y_train_bal)
print('Done.')

from sklearn.linear_model import LogisticRegression

prob_model = LogisticRegression(
    max_iter=2000, multi_class='multinomial',
    C=1.0, solver='lbfgs', random_state=42,
)
print('Training probability calibrator (LogisticRegression)...')
prob_model.fit(X_train_bal, y_train_bal)

prob_preds = prob_model.predict(X_test)
prob_f1 = f1_score(y_test_cls, prob_preds, average='macro')
prob_acc = accuracy_score(y_test_cls, prob_preds)
print(f'Done — LogReg F1={prob_f1:.4f}, Acc={prob_acc:.4f}')

bw_mask = y_test_cls == 0
bw_probas = prob_model.predict_proba(X_test[bw_mask])[:, 0]
print(f'BW probability range: {bw_probas.min():.4f} — {bw_probas.max():.4f} (mean {bw_probas.mean():.4f})')

In [ ]:
y_pred_cls = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)

f1 = f1_score(y_test_cls, y_pred_cls, average='macro')
acc = accuracy_score(y_test_cls, y_pred_cls)

print('='*60)
print('FAILURE CLASSIFIER — Test Set Results')
print('='*60)
print(classification_report(y_test_cls, y_pred_cls, target_names=le.classes_))
print(f'Macro F1:  {f1:.4f}')
print(f'Accuracy:  {acc:.4f}')

In [ ]:
cm = confusion_matrix(y_test_cls, y_pred_cls)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(le.classes_)))
ax.set_yticks(range(len(le.classes_)))
ax.set_xticklabels(le.classes_, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(le.classes_, fontsize=9)
for i in range(len(le.classes_)):
    for j in range(len(le.classes_)):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=11)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Failure Classifier — Confusion Matrix')
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
cv_scores = cross_val_score(clf, X_train_bal, y_train_bal, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42), scoring='f1_macro', n_jobs=1)
print(f'3-Fold Cross-Validation F1 (macro): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Fold scores: {["{:.4f}".format(s) for s in cv_scores]}')

---
## 5b. Probability Calibration — Temperature Scaling

XGBoost with `multi:softprob` tends to produce **overconfident probabilities** — outputs near 0 or 1 even for moderate degradation. This makes it impossible to visualize gradual failure progression over time.

**Temperature scaling** (Guo et al., 2017) is a post-hoc calibration technique that divides the model's raw logit margins by a temperature parameter `T > 1` before applying softmax. This recovers the confidence gradient encoded in the logits without changing the predicted class.

- `T = 1.0` → uncalibrated (original model output)
- `T > 1.0` → softer, more gradual probabilities
- Argmax is preserved — the predicted failure mode doesn't change

In [ ]:
import xgboost as xgb
from scipy.optimize import minimize_scalar

dmat_test = xgb.DMatrix(X_test, feature_names=FEATURE_COLS)
raw_margins = clf.get_booster().predict(dmat_test, output_margin=True)

def temperature_softmax(margins, T):
    scaled = margins / T
    exp_m = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_m / exp_m.sum(axis=1, keepdims=True)

def nll_loss(T):
    probs = temperature_softmax(raw_margins, T)
    correct_probs = probs[np.arange(len(y_test_cls)), y_test_cls]
    return -np.log(correct_probs + 1e-10).mean()

result = minimize_scalar(nll_loss, bounds=(0.5, 20.0), method='bounded')
optimal_T = round(result.x, 2)

print(f'Optimal temperature (NLL-minimized): T = {optimal_T}')
print(f'NLL at T=1.0 (uncalibrated): {nll_loss(1.0):.4f}')
print(f'NLL at T={optimal_T} (calibrated): {nll_loss(optimal_T):.4f}')
print()

failure_mask = y_test_cls != le.transform(['NORMAL'])[0]
for T_val in [1.0, 2.0, optimal_T, 5.0, 8.0]:
    probs = temperature_softmax(raw_margins[failure_mask], T_val)
    max_probs = probs.max(axis=1)
    print(f'  T={T_val:>5.1f}  |  failure samples avg max prob: {max_probs.mean():.4f}  '
          f'(range {max_probs.min():.4f} — {max_probs.max():.4f})')

CALIBRATION_TEMP = optimal_T
print(f'\nUsing CALIBRATION_TEMP = {CALIBRATION_TEMP} for fleet scoring')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Effect of Temperature Scaling on Probability Distributions', fontweight='bold')

for ax, T_val in zip(axes, [1.0, CALIBRATION_TEMP, 8.0]):
    probs = temperature_softmax(raw_margins[failure_mask], T_val)
    ax.hist(probs.max(axis=1), bins=30, color='#4f46e5', alpha=0.7, edgecolor='white')
    ax.set_xlabel('Max Class Probability')
    ax.set_ylabel('Count')
    ax.set_title(f'T = {T_val:.1f}')
    ax.set_xlim(0, 1.05)
    ax.axvline(0.5, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. RUL Regressor — Remaining Useful Life

Train a separate XGBoost regressor on non-normal samples to predict days until failure. Only assets with active degradation patterns get RUL predictions — healthy assets receive a default high RUL score.

In [ ]:
normal_idx = le.transform(['NORMAL'])[0]
nn_train = y_train_cls != normal_idx
nn_test = y_test_cls != normal_idx

print(f'RUL training samples: {nn_train.sum()}')
print(f'RUL test samples:     {nn_test.sum()}')
print(f'RUL range (train): {y_train_rul[nn_train].min():.1f} — {y_train_rul[nn_train].max():.1f} days')

reg = XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=2.0,
    random_state=42, tree_method='hist'
)
reg.fit(X_train[nn_train], y_train_rul[nn_train])

y_pred_rul = reg.predict(X_test[nn_test])
mae = mean_absolute_error(y_test_rul[nn_test], y_pred_rul)
rmse = root_mean_squared_error(y_test_rul[nn_test], y_pred_rul)
r2 = r2_score(y_test_rul[nn_test], y_pred_rul)

print(f'\n{"="*60}')
print('RUL REGRESSOR — Test Set Results')
print(f'{"="*60}')
print(f'MAE:  {mae:.2f} days')
print(f'RMSE: {rmse:.2f} days')
print(f'R²:   {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_rul[nn_test], y_pred_rul, alpha=0.6, s=30, c='#4f46e5')
lims = [0, max(y_test_rul[nn_test].max(), y_pred_rul.max()) + 2]
axes[0].plot(lims, lims, 'r--', alpha=0.5, label='Perfect prediction')
axes[0].set_xlabel('Actual RUL (days)')
axes[0].set_ylabel('Predicted RUL (days)')
axes[0].set_title(f'RUL: Predicted vs Actual (R²={r2:.3f})')
axes[0].legend()

residuals = y_test_rul[nn_test] - y_pred_rul
axes[1].hist(residuals, bins=20, color='#4f46e5', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Residual (Actual - Predicted, days)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'RUL Residual Distribution (MAE={mae:.2f}d)')

plt.tight_layout()
plt.show()

---
## 7. Model Explainability — SHAP Analysis

Use SHAP (SHapley Additive exPlanations) to understand which features drive failure predictions. This is critical for:
- **Trust** — Technicians need to understand *why* a failure is predicted
- **Actionability** — Knowing the root cause sensor helps target inspection
- **Compliance** — Audit trail for maintenance decisions

In [ ]:
# SHAP analysis skipped - version incompatibility in Snowflake Notebooks
# To run locally: uncomment and execute
# import shap
# explainer = shap.TreeExplainer(clf)
# shap_values = explainer.shap_values(X_test)
print('SHAP analysis skipped (run locally if needed)')

In [ ]:
# SHAP per-class analysis skipped
print('SHAP per-class analysis skipped')

In [ ]:
# SHAP RUL analysis skipped
print('SHAP RUL analysis skipped')

---
## 8. Register Models in Snowflake ML Registry

Register both models with metrics, sample input data, and metadata for governance and reproducibility.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name='PDM_DEMO', schema_name='ML')

X_df = pd.DataFrame(X_test, columns=FEATURE_COLS)

clf_mv = registry.log_model(
    model=clf,
    model_name='FAILURE_CLASSIFIER',
    version_name='v6',
    sample_input_data=X_df.head(10),
    conda_dependencies=['xgboost', 'scikit-learn'],
    metrics={
        'f1_macro': round(float(f1), 4),
        'accuracy': round(float(acc), 4),
        'cv_f1_mean': round(float(cv_scores.mean()), 4),
        'cv_f1_std': round(float(cv_scores.std()), 4),
        'n_classes': len(le.classes_),
        'n_features': len(FEATURE_COLS),
        'train_samples': int(X_train_bal.shape[0]),
        'smote_applied': True,
    },
    comment='Multi-class failure classifier (XGBoost + SMOTE). Severity features for calibrated probability gradients. '
            f'Classes: {", ".join(le.classes_)}.'
)
print(f'FAILURE_CLASSIFIER v6 registered — F1={f1:.4f}, Acc={acc:.4f}')

X_rul_df = pd.DataFrame(X_test[nn_test], columns=FEATURE_COLS)

reg_mv = registry.log_model(
    model=reg,
    model_name='RUL_REGRESSOR',
    version_name='v6',
    sample_input_data=X_rul_df.head(10),
    conda_dependencies=['xgboost', 'scikit-learn'],
    metrics={
        'mae_days': round(float(mae), 2),
        'rmse_days': round(float(rmse), 2),
        'r2': round(float(r2), 4),
        'n_features': len(FEATURE_COLS),
        'train_samples': int(nn_train.sum()),
    },
    comment=f'RUL regressor (XGBoost). MAE={mae:.2f}d, RMSE={rmse:.2f}d, R²={r2:.4f}. '
            'Trained on non-normal samples only.'
)
print(f'RUL_REGRESSOR v6 registered — MAE={mae:.2f}d, R²={r2:.4f}')

prob_mv = registry.log_model(
    model=prob_model,
    model_name='PROBABILITY_CALIBRATOR',
    version_name='v6',
    sample_input_data=X_df.head(10),
    conda_dependencies=['scikit-learn'],
    metrics={
        'f1_macro': round(float(prob_f1), 4),
        'accuracy': round(float(prob_acc), 4),
        'n_features': len(FEATURE_COLS),
    },
    comment='LogisticRegression probability calibrator — produces severity-scaled class probabilities '
            'that increase as degradation features worsen. Trained on SMOTE-balanced data.',
)
print(f'PROBABILITY_CALIBRATOR v6 registered — F1={prob_f1:.4f}, Acc={prob_acc:.4f}')

In [ ]:
print('Models in registry:')
for m in registry.models():
    print(f'  {m.name}')
    for v in m.versions():
        print(f'    {v.version_name}')

---
## Summary

### Pipeline Components
- **SMOTE** oversampling to handle 98% class imbalance (6 classes)
- **RandomizedSearchCV** (40 iterations × 5 folds = 200 fits) for hyperparameter optimization
- **SHAP** explainability — global importance + per-failure-mode feature drivers
- **Snowflake ML Registry** for model governance, versioning, and metrics tracking
- **No batch inference here** — run `score_fleet.ipynb` to load these models from the registry and score the fleet

### Registered Models
| Model | Registry Path | Key Metric |
|-------|--------------|------------|
| Failure Classifier | `PDM_DEMO.ML.FAILURE_CLASSIFIER` (v2) | Macro F1 (see above) |
| RUL Regressor | `PDM_DEMO.ML.RUL_REGRESSOR` (v2) | MAE in days (see above) |

### Feature Engineering
6 domain-specific features engineered from raw sensor aggregations:
- `VIB_TEMP_INTERACTION` — Bearing wear signature (vibration × temperature)
- `POWER_EFFICIENCY` — Power draw per unit flow rate
- `PRESSURE_VARIABILITY` — Coefficient of variation for pressure
- `VIB_DEVIATION` / `TEMP_DEVIATION` — Gap between max and mean (intermittent spikes)
- `MAINT_RECENCY_SCORE` — Inverse of days since last maintenance

Both models are now registered in the Snowflake Model Registry at `PDM_DEMO.ML`.\n",
    "Run `score_fleet.ipynb` to load these models and generate fleet-wide predictions.